In [1]:
from tqdm.notebook import tqdm
from pymatgen.core import Structure, Element, Composition
from pymatgen.analysis.phase_diagram import PhaseDiagram, PDEntry
from pymatgen.ext.matproj import MPRester
import os
import pandas as pd
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

import re
import warnings
from matplotlib import pyplot as plt

from pymatgen.io.vasp.sets import MPMetalRelaxSet
from pymatgen.io.ase import AseAtomsAdaptor as aaa

warnings.filterwarnings('ignore')
tqdm.pandas()

In [2]:
def is_metal(df):
    return any([df[method] <0.1 for method in ['PBE','GLLB-SC','HSE','SCAN']])

def get_sgn(df):
    analyzer = SpacegroupAnalyzer(df.structure)
    sym_data = analyzer.get_symmetry_dataset()
    return sym_data['number']

def get_comp(df):
    return df.structure.composition.reduced_formula

In [4]:
par_el = 'Be'
root_relaxed = f'/blue/hennig/jasongibson/elemental_sub/materials/{par_el}/m3gnet_relaxed'
df = pd.read_pickle(f'/blue/hennig/jasongibson/elemental_sub/pickle_files/df_{par_el}_pred_m3gnet.pkl')
df = df.loc[df.tcad>5]

In [103]:
# Function to sort elements in a chemical formula
def sort_elements(formula):
    # Extract elements using a regular expression
    elements = re.findall(r'[A-Z][a-z]*', formula)
    # Sort the elements alphabetically
    sorted_elements = ''.join(sorted(elements))
    return sorted_elements

# Apply the function to create a new column with sorted elements
df['Sorted_Elements'] = df['formula'].apply(sort_elements)

# Sort the dataframe by the sorted elements
df.sort_values(by='Sorted_Elements',inplace=True)


In [106]:
inds = []
eahs = []
cur_elements = {}
with MPRester('7JGUQgNZyOTTp8Tc') as mpr:
    for i in tqdm(df.index[:1]):
        if os.path.isfile(root_relaxed+f'/POSCAR_{i}'):
            entry = df.loc[i]
            struct = Structure.from_file(root_relaxed+f'/POSCAR_{i}')
            # species =list(set([spec.name for spec in struct.species]))
            elements = struct.symbol_set
            if cur_elements != elements:
                entries = mpr.get_entries_in_chemsys(elements)
                cur_elements = elements
                phasediagram = PhaseDiagram(entries)

            
            final_E = entry['final_E']
            pde = PDEntry(struct.composition,final_E)

            eah = phasediagram.get_e_above_hull(pde,allow_negative=True)
            eahs.append(eah)
            inds.append(i)

  0%|          | 0/1 [00:00<?, ?it/s]

In [73]:
df_eah = df.loc[inds]
df_eah['eah'] = eahs


In [76]:
df_eah.to_pickle(f'pickle_files/df_{par_el}_pred_m3gnet_all.pkl')
df_eah = df_eah.loc[df_eah.eah<=0.2]#.head(25)#.sort_values('eah')
df_eah.to_pickle(f'pickle_files/df_{par_el}_pred_m3gnet.pkl')

In [7]:
df_eah.sort_values('tcad',inplace=True,ascending=False)

In [8]:
def get_natoms(df):
    return len(df.structure)
df_eah['natoms'] = df_eah.apply(get_natoms,axis=1)

In [12]:
data_path = '/blue/hennig/ajinkya.hire/superC/elemental_alpha2F/PBEsol/Sampling_the_materials_space_for_conventional_superconducting_compounds/qe_data_imaginary_removed.pkl'
df_train = pd.read_pickle(data_path)

In [14]:
comps = df_train.comp.values
df_eah = df_eah.loc[~df_eah.formula.isin(comps)]#.shape
df_eah.shape

(4086, 20)

In [26]:
df_mp = pd.read_pickle(f'pickle_files/df_mp_data_mp_pred.pkl')
comps = df_mp.formula.values
df_eah = df_eah.loc[~df_eah.formula.isin(comps)]#.shape
df_eah.shape

(4023, 20)

In [8]:
df_eah.sort_values('tcad',inplace=True,ascending=False)


In [30]:
root_mp = f'/blue/hennig/jasongibson/elemental_sub/materials/{par_el}/mp_relaxed/'

In [43]:
for index, row in tqdm(df_eah.iterrows(),total=len(df_eah)):
    structure = aaa.get_structure(row.structure)
    mprs = MPMetalRelaxSet(structure)
    mprs.write_input(root_mp + f'{index}')      

  0%|          | 0/4023 [00:00<?, ?it/s]